# Run the following block of code to prepare shap (This is a bonus question)

In [ ]:
!pip install -q shap sentencepiece
import torch
from transformers import pipeline

import shap


# load the model
model_qa = "distilbert-base-uncased-distilled-squad" # current default model
pmodel = pipeline("question-answering", model=model_qa)

# define two predictions, one that outputs the logits for the range start,
# and the other for the range end
def f(questions, start):
    outs = []
    for q in questions:
        question, context = q.split("[SEP]")
        #print('-'*40)
        #print(f'Q: {question}\nC:: {context}')
        d = pmodel.tokenizer(question, context) # d = {'input_ids', 'attention_mask'}
        out = pmodel.model.forward(**{k: torch.tensor(d[k]).reshape(1, -1) for k in d})
        logits = out.start_logits if start else out.end_logits
        outs.append(logits.reshape(-1).detach().numpy())
    return outs


def f_start(questions):
    return f(questions, True)


def f_end(questions):
    return f(questions, False)


# attach a dynamic output_names property to the models so we can plot the tokens at each output position
def out_names(inputs):
    question, context = inputs.split("[SEP]")
    d = pmodel.tokenizer(question, context)
    return [pmodel.tokenizer.decode([id]) for id in d["input_ids"]]


f_start.output_names = out_names
f_end.output_names = out_names

# Bonus Q: Change the context in a way that the answer mentions alice.
- You are not allowed to remove the following part of the context: 'snoop dogg'
- You can not change the question.
- You may consider using shap plot to better understand the model decision.

In [ ]:
# Now you can use the pipeline to answer questions
context = 'Alice opened the cage of her german shepherd while ice cube let the snoop dogg out of jail.'
question = 'Who let the dog out?'
answer = pmodel({
    'context': context,
    'question': question
})

print(answer)

In [ ]:
# a simple dataset
data = [
    f"{question}[SEP]{context}"
]

explainer_start = shap.Explainer(f_start, pmodel.tokenizer)
shap_values_start = explainer_start(data)

shap.plots.text(shap_values_start)